# TP1 — Équations normales : densité–porosité (pétrole)

[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/earthai-tech/lab-courses/blob/main/inphb/prepa/notebooks/PREPA_TP1_equations_normales.ipynb)
Objectif : estimer ρ_matrice et ρ_fluide par moindres carrés, vérifier l’orthogonalité des résidus.

> Données : dossier `PREPA_TP_donnees`. Règle du module : **numpy uniquement**, pas de bibliothèque d’apprentissage.

In [ ]:
# Configuration : Colab ou Jupyter local. Exécutez cette cellule en premier.
import os, shutil, urllib.request
DONNEES = ['petro_densite_porosite.csv']
URL = 'https://raw.githubusercontent.com/earthai-tech/lab-courses/main/inphb/prepa/data/'
for f in DONNEES:
    if os.path.exists(f):
        continue
    if os.path.exists(os.path.join('..', 'data', f)):
        shutil.copy(os.path.join('..', 'data', f), f)   # kit hors ligne / dépôt cloné
    else:
        urllib.request.urlretrieve(URL + f, f)            # Colab : téléchargement
    print('OK', f)


In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
d = pd.read_csv('petro_densite_porosite.csv')
phi, rho = d['porosite'].values, d['densite_g_cc'].values
n = len(phi); print(n, 'mesures')

## 1. Matrice de conception
À COMPLÉTER : construire X (colonne de 1, colonne φ).

In [ ]:
X = np.c_[np.ones(n), phi]
print(X.shape)

## 2. Résoudre les équations normales  XᵀX θ = Xᵀy

In [ ]:
theta = np.linalg.solve(X.T @ X, X.T @ rho)
rho_m, rho_f = theta[0], theta[0] + theta[1]
print(f'rho_m = {rho_m:.3f} g/cm3 ; rho_f = {rho_f:.3f} g/cm3')

## 3. Vérifications
- Orthogonalité : Xᵀ(y − Xθ) ≈ 0
- Comparaison avec `lstsq` et avec l’inverse explicite

In [ ]:
res = rho - X @ theta
print('X^T res =', X.T @ res)
R2 = 1 - res @ res / np.sum((rho - rho.mean())**2)
print('R2 =', round(R2, 3))
th2 = np.linalg.lstsq(X, rho, rcond=None)[0]
th3 = np.linalg.inv(X.T @ X) @ X.T @ rho
print(np.allclose(theta, th2), np.allclose(theta, th3))
print('cond(X) =', np.linalg.cond(X), ' cond(XtX) =', np.linalg.cond(X.T @ X))

In [ ]:
plt.scatter(phi, rho, s=20)
xs = np.linspace(0, .35, 10); plt.plot(xs, theta[0] + theta[1] * xs, 'r')
plt.xlabel('porosité'); plt.ylabel('densité'); plt.show()
plt.scatter(phi, res); plt.axhline(0, color='r'); plt.title('Résidus'); plt.show()

## Questions
1. Les valeurs de ρ_m et ρ_f sont-elles physiquement plausibles ?
2. Vérifier numériquement cond(XᵀX) = cond(X)².
3. Ajouter un bruit 10 fois plus fort sur ρ : comment évoluent θ et R² ?